In [20]:
import os
import cv2
import json
import joblib
import numpy as np
import mediapipe as mp
from collections import deque
import warnings
from tensorflow.keras.models import load_model
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [21]:
MODEL_PATH = "hand_gesture_mlp.h5"
SCALER_PATH = "scaler.pkl"
ENCODER_PATH = "label_encoder.pkl"
GM_PATH = "gesture_map.json"

for path in [MODEL_PATH, SCALER_PATH, ENCODER_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Không tìm thấy '{path}'. Hãy train mô hình trước!")

In [22]:
model = load_model(MODEL_PATH)          # Dùng Keras để load file .h5
scaler = joblib.load(SCALER_PATH)
le = joblib.load(ENCODER_PATH)
label_names = list(le.classes_)

In [23]:
LABEL_DESCRIPTIONS = {}
if os.path.exists(GM_PATH):
    try:
        with open(GM_PATH, "r", encoding="utf-8") as f:
            gm = json.load(f)
        for v in gm.values():
            if isinstance(v, dict) and "label" in v:
                LABEL_DESCRIPTIONS[v["label"]] = v.get("desc", "")
    except Exception:
        pass


In [24]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

In [25]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

True

In [26]:
SMOOTH_WIN = 5
UNKNOWN_THRESHOLD = 0.4
proba_buffer = deque(maxlen=SMOOTH_WIN)

In [27]:
try:
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.7,
        min_tracking_confidence=0.5,
    ) as hands:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)

            display_text = "Nhấn 'q' để thoát"
            gesture_name_display = "Unknown"
            confidence = 0.0
            top_info = None

            if result.multi_hand_landmarks:
                hand_landmarks = result.multi_hand_landmarks[0]
                mp_draw.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_draw.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=3),
                    mp_draw.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                )

                # Trích xuất landmark
                data_point = []
                for lm in hand_landmarks.landmark:
                    data_point.extend([lm.x, lm.y, lm.z])

                if len(data_point) == 63:
                    X = np.array(data_point, dtype=np.float32).reshape(1, -1)
                    X = scaler.transform(X)

                    # ==========================
                    # 🔮 DỰ ĐOÁN BẰNG MLP (.h5)
                    # ==========================
                    probs = model.predict(X, verbose=0)[0]  # (n_class,)
                    proba_buffer.append(probs)
                    probs_smoothed = np.mean(proba_buffer, axis=0)

                    pred_class_idx = np.argmax(probs_smoothed)
                    gesture_name = le.inverse_transform([pred_class_idx])[0]
                    confidence = float(probs_smoothed[pred_class_idx]) * 100

                    top_idx = np.argsort(probs_smoothed)[::-1][:3]
                    top_info = [(label_names[i], float(probs_smoothed[i])) for i in top_idx]

                    if probs_smoothed[pred_class_idx] < UNKNOWN_THRESHOLD:
                        gesture_name_display = "Unknown"
                    else:
                        gesture_name_display = gesture_name

                    display_text = f"Gesture: {gesture_name_display} ({confidence:.1f}%)"
                else:
                    display_text = "Không đủ landmark (63)"
            else:
                display_text = "Không phát hiện bàn tay"

            # ==========================
            # 💬 HIỂN THỊ LÊN MÀN HÌNH
            # ==========================
            cv2.putText(frame, display_text, (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            desc = LABEL_DESCRIPTIONS.get(gesture_name_display)
            if desc:
                cv2.putText(frame, desc, (10, 70),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)

            # Vẽ thanh top-3 xác suất
            if top_info:
                BAR_X, BAR_Y, BAR_W, BAR_H, GAP = 10, 110, 300, 20, 10
                for i, (name, p) in enumerate(top_info):
                    y = BAR_Y + i * (BAR_H + GAP)
                    cv2.rectangle(frame, (BAR_X, y), (BAR_X + BAR_W, y + BAR_H), (50, 50, 50), -1)
                    cv2.rectangle(frame, (BAR_X, y), (BAR_X + int(BAR_W * p), y + BAR_H),
                                  (0, 255 - i * 60, 255), -1)
                    cv2.putText(frame, f"{name}: {p*100:.1f}%",
                                (BAR_X + BAR_W + 10, y + BAR_H - 4),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

            cv2.imshow("🖐 Real-time Gesture Recognition", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()